# 🍏 Apple Generative Imagery Systems - FLUX.1 [dev] LoRA Training Pipeline
### Target: FLUX.1-dev (12B MMDiT) | Dataset: 21 Curated Apple Minimal LookDev Assets
---
이 주피터 노트북은 Google Colab(T4 16GB / L4 24GB / A100) 환경에서 **Apple 스탠다드 미니멀 미학(무광 아노다이징 알루미늄, CMF 디테일, 앰비언트 오클루전 라이팅)**을 가진 **FLUX.1-dev LoRA**를 학습하는 올인원 자동화 파이프라인입니다.

**핵심 기술 사양:**
- **엔진**: Ostris AI-Toolkit (FLUX 공식 표준 LoRA 학습 프레임워크)
- **아키텍처**: 12B Flow Matching Transformer (MMDiT)
- **최적화**: FP8 Base Model Caching + 8-bit AdamW Optimizer + Gradient Checkpointing (Colab T4 16GB 완벽 구동 지원)
- **트리거 토큰**: `apple minimal craft style`, `clean matte studio lookdev`

In [2]:
# 1. GPU 하드웨어 가속기 사양 확인 (Colab 런타임 유형이 GPU로 되어 있는지 확인)
!nvidia-smi

Sun Sep  6 02:44:40 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   32C    P0             45W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [4]:
# 2. Google Drive 마운트
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [6]:
# [Cell 3] FLUX AI-Toolkit 깨끗하게 재설치
import shutil, os

# 기존 충돌 폴더 깔끔하게 삭제 후 클론
if os.path.exists('/content/ai-toolkit'):
    shutil.rmtree('/content/ai-toolkit')

%cd /content
!git clone https://github.com/ostris/ai-toolkit.git /content/ai-toolkit
%cd /content/ai-toolkit
!git submodule update --init --recursive

# 필수 의존성 패키지 설치
!pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu121
!pip install -q -r requirements.txt
!pip install -q oyaml diffusers transformers albumentations invisible-watermark bitsandbytes accelerate huggingface_hub

print("✅ AI-Toolkit 및 필수 패키지 설치 완료!")


/content
Cloning into '/content/ai-toolkit'...
remote: Enumerating objects: 13615, done.
remote: Counting objects: 100% (6537/6537), done.
remote: Compressing objects: 100% (929/929), done.
remote: Total 13615 (delta 5995), reused 5608 (delta 5608), pack-reused 7078 (from 3)
Receiving objects: 100% (13615/13615), 34.39 MiB | 19.16 MiB/s, done.
Resolving deltas: 100% (9471/9471), done.
/content/ai-toolkit
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.5/96.5 kB 10.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.1/63.1 kB 7.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.8/56.8 MB 45.3 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  error: subprocess-exited-with-error
  
  

In [7]:
# 4. Hugging Face 로그인 (FLUX.1-dev 모델 가중치 접근용 Token 입력)
# https://huggingface.co/settings/tokens 에서 발급받은 Access Token을 입력하세요.
from huggingface_hub import login
import getpass

print('🔑 Hugging Face Access Token을 입력하세요 (FLUX.1-dev 동의 완료된 계정):')
hf_token = getpass.getpass('HF Token: ')
login(token=hf_token)
print('✅ Hugging Face 로그인 완료!')

🔑 Hugging Face Access Token을 입력하세요 (FLUX.1-dev 동의 완료된 계정):
HF Token: ··········
✅ Hugging Face 로그인 완료!


In [10]:
# [Cell 5] FLUX 전용 21장 정제 데이터셋(flux_train_1024) 배치
import os, shutil

DATASET_DIR = '/content/dataset'
if os.path.exists(DATASET_DIR):
    shutil.rmtree(DATASET_DIR)
os.makedirs(DATASET_DIR, exist_ok=True)

# GitHub 레포지토리에서 최신 FLUX 전용 데이터셋 복제
if os.path.exists('/content/repo_temp'):
    shutil.rmtree('/content/repo_temp')

print('🌐 GitHub에서 최신 21장 FLUX 전용 데이터셋(flux_train_1024) 다운로드 중...')
!git clone https://github.com/djbyun/spatial-gen-pipeline.git /content/repo_temp

flux_source = '/content/repo_temp/dataset/flux_train_1024'
for f in os.listdir(flux_source):
    if f.endswith('.png') or f.endswith('.txt'):
        shutil.copy2(os.path.join(flux_source, f), os.path.join(DATASET_DIR, f))

# 첫 번째 캡션 확인
sample_txt = open(os.path.join(DATASET_DIR, 'image_00.txt'), 'r').read()
img_count = len([f for f in os.listdir(DATASET_DIR) if f.endswith('.png')])
txt_count = len([f for f in os.listdir(DATASET_DIR) if f.endswith('.txt')])

print(f'\n✅ 100% FLUX 전용 데이터셋 준비 완료!')
print(f'📊 이미지: {img_count}장 | 캡션: {txt_count}개')
print(f'📝 첫 번째 캡션 검증: "{sample_txt.strip()}"')


🌐 GitHub에서 최신 21장 FLUX 전용 데이터셋(flux_train_1024) 다운로드 중...
Cloning into '/content/repo_temp'...
remote: Enumerating objects: 262, done.
remote: Counting objects: 100% (78/78), done.
remote: Compressing objects: 100% (60/60), done.
remote: Total 262 (delta 19), reused 72 (delta 17), pack-reused 184 (from 2)
Receiving objects: 100% (262/262), 230.41 MiB | 17.44 MiB/s, done.
Resolving deltas: 100% (56/56), done.

✅ 100% FLUX 전용 데이터셋 준비 완료!
📊 이미지: 21장 | 캡션: 21개
📝 첫 번째 캡션 검증: "a photo in apple minimal craft style of, clean matte studio lookdev, organic cosmetic cream swirls, viscous pastel orange and beige cosmetic texture, soft studio diffuse lighting, macro close-up view, creamy tactile surface, neutral studio background"


In [15]:
# 6. FLUX LoRA 학습용 최적화 설정 파일 (YAML) 생성 - A100 40GB 모드
import os

config_yaml_content = """
job: extension
config:
  name: "apple_minimal_craft_flux_v1"
  process:
    - type: 'sd_trainer'
      training_folder: "/content/output_flux_lora"
      device: cuda:0
      trigger_word: "apple minimal craft style"
      network:
        type: "lora"
        linear: 16
        linear_alpha: 16
      save:
        dtype: float16
        save_every: 250
        max_step_saves_to_keep: 4
      datasets:
        - folder_path: "/content/dataset"
          caption_ext: "txt"
          caption_dropout_rate: 0.05
          shuffle_tokens: false
          cache_latents_to_disk: true
          resolution: [1024]
      train:
        batch_size: 1
        steps: 1200
        gradient_accumulation_steps: 1
        train_unet: true
        train_text_encoder: false
        gradient_checkpointing: true
        noise_scheduler: "flowmatch"
        optimizer: "adamw8bit"
        lr: 1e-4
        ema_config:
          use_ema: false
        dtype: bf16
      model:
        name_or_path: "black-forest-labs/FLUX.1-dev"
        is_flux: true
        quantize: false
        low_vram: false
      sample:
        sampler: "flowmatch"
        sample_every: 250
        width: 1024
        height: 1024
        prompts:
          - "apple minimal craft style, clean matte studio lookdev, anodized aluminum cylinder on neutral white podium, soft studio lighting"
          - "apple minimal craft style, organic curved frosted glass object, ambient occlusion lighting, neutral background"
        neg: ""
        seed: 42
        walk_seed: true
        guidance_scale: 3.5
        sample_steps: 25
"""

os.makedirs('/content/ai-toolkit/config', exist_ok=True)
with open('/content/ai-toolkit/config/apple_craft_flux.yaml', 'w') as f:
    f.write(config_yaml_content.strip())

print('✅ A100 네이티브 BF16 설정 파일(/content/ai-toolkit/config/apple_craft_flux.yaml) 생성 완료!')


✅ A100 네이티브 BF16 설정 파일(/content/ai-toolkit/config/apple_craft_flux.yaml) 생성 완료!


In [19]:
# 7. FLUX.1-dev LoRA 파인튜닝 실행 (A100 기준 약 7~10분 소요)
import torch

torch.cuda.empty_cache()

%cd /content/ai-toolkit
!pip install -q controlnet_aux opencv-python einops kornia lycoris-lora toml optimum optimum-quanto diffusers transformers accelerate oyaml albumentations safetensors invisible-watermark bitsandbytes lpips av ftfy timm
!python run.py config/apple_craft_flux.yaml


/content/ai-toolkit
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 290.4/290.4 kB 23.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 81.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 133.6 MB/s eta 0:00:00
Multiple distributions found for package optimum. Picked distribution: optimum
Running 1 job
{
    "type": "sd_trainer",
    "training_folder": "/content/output_flux_lora",
    "device": "cuda:0",
    "trigger_word": "apple minimal craft style",
    "network": {
        "type": "lora",
        "linear": 16,
        "linear_alpha": 16
    },
    "save": {
        "dtype": "float16",
        "save_every": 250,
        "max_step_saves_to_keep": 4
    },
    "datasets": [
        {
            "folder_path": "/content/dataset",
            "caption_ext": "txt",
            "caption_dropout_rate": 0.05,
            "shuffle_tokens": false,
            "cache_latents_to_disk": true,
            "resolution": [
                1024
         

In [20]:
# [Cell 8] 최종 LoRA 가중치(.safetensors) 및 샘플 렌더링 Google Drive / 로컬 백업
import os, glob, shutil
from google.colab import files

DEST_DIR = '/content/drive/MyDrive/spatial-gen-pipeline/weights/flux'
SAMPLES_DIR = '/content/drive/MyDrive/spatial-gen-pipeline/samples/flux'
os.makedirs(DEST_DIR, exist_ok=True)
os.makedirs(SAMPLES_DIR, exist_ok=True)

# 1. 모든 LoRA 체크포인트 가중치 백업
saved_loras = glob.glob('/content/output_flux_lora/**/*.safetensors', recursive=True)
if saved_loras:
    for lora_path in saved_loras:
        fname = os.path.basename(lora_path)
        shutil.copy2(lora_path, os.path.join(DEST_DIR, fname))
        print(f'✅ LoRA 가중치 백업 완료: {fname}')

    # 최종 1200스텝 LoRA 파일 로컬 PC 다운로드
    final_lora = '/content/output_flux_lora/apple_minimal_craft_flux_v1/apple_minimal_craft_flux_v1.safetensors'
    if not os.path.exists(final_lora):
        final_lora = sorted(saved_loras)[-1]

    print(f'\n📥 최종 LoRA 파일 로컬 PC 다운로드 시작: {os.path.basename(final_lora)}')
    files.download(final_lora)
else:
    print('⚠️ 가중치 파일을 찾지 못했습니다.')

# 2. 250스텝마다 자동 생성된 샘플 렌더 이미지 백업
sample_imgs = glob.glob('/content/output_flux_lora/**/*.png', recursive=True) + glob.glob('/content/output_flux_lora/**/*.jpg', recursive=True)
if sample_imgs:
    for img_path in sample_imgs:
        fname = os.path.basename(img_path)
        shutil.copy2(img_path, os.path.join(SAMPLES_DIR, fname))
    print(f'\n🖼️ 250스텝별 샘플 이미지 {len(sample_imgs)}장 Google Drive 백업 완료!')


✅ LoRA 가중치 백업 완료: apple_minimal_craft_flux_v1.safetensors
✅ LoRA 가중치 백업 완료: apple_minimal_craft_flux_v1_000000250.safetensors
✅ LoRA 가중치 백업 완료: apple_minimal_craft_flux_v1_000000750.safetensors
✅ LoRA 가중치 백업 완료: apple_minimal_craft_flux_v1_000000500.safetensors
✅ LoRA 가중치 백업 완료: apple_minimal_craft_flux_v1_000001000.safetensors

📥 최종 LoRA 파일 로컬 PC 다운로드 시작: apple_minimal_craft_flux_v1.safetensors


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


🖼️ 250스텝별 샘플 이미지 12장 Google Drive 백업 완료!
